# Day, Wind, Pressure, Temperature, and Humidity Follow-Up at Minute Maid Park

**Goals:**
- Investigate how **day vs night** changes run scoring at Minute Maid Park.
- Measure how **wind speed, pressure, temperature, and humidity** each line up with scoring.
- Check whether any day/night scoring gap still holds inside similar **weather quintiles**.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 220)

TEAM_CODE = 'HOU'

params = pd.read_csv('../team_parameters.csv')
team_meta = params[params['team_code'] == TEAM_CODE].iloc[0]

team_df = pd.read_csv(f"../../data/{team_meta['dataset_file']}")
team_df = team_df[(team_df['season'] >= int(team_meta['data_start_year'])) & (team_df['season'] <= int(team_meta['data_end_year']))].copy()
team_df = team_df.dropna(subset=['start_hour', 'total_runs', 'home_runs_scored', 'away_runs_scored', 'temp_f', 'rhum', 'pres', 'wspd_mph'])

team_df['game_date'] = pd.to_datetime(team_df['game_date'])
team_df['month'] = team_df['game_date'].dt.month
month_map = {3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun', 7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct'}
team_df['month_name'] = team_df['month'].map(month_map)
team_df['time_of_day'] = np.where(team_df['start_hour'] < 17, 'Day', 'Night')

team_df['temp_bin'] = pd.qcut(team_df['temp_f'], q=5, duplicates='drop')
team_df['rhum_bin'] = pd.qcut(team_df['rhum'], q=5, duplicates='drop')
team_df['pres_bin'] = pd.qcut(team_df['pres'], q=5, duplicates='drop')
team_df['wspd_bin'] = pd.qcut(team_df['wspd_mph'], q=5, duplicates='drop')

print(f"Dataset: {team_meta['dataset_file']} ({int(team_meta['data_start_year'])}-{int(team_meta['data_end_year'])})")
print(f"Total {TEAM_CODE} home games with complete day/night + weather data: {len(team_df)}")
print('\nDay/Night counts:')
print(team_df['time_of_day'].value_counts())

In [ ]:
# ============================================================
# Day vs Night Summary Table
# ============================================================
day_night_summary = team_df.groupby('time_of_day', observed=True).agg(
    games=('game_pk', 'size'),
    total_runs_mean=('total_runs', 'mean'),
    home_runs_mean=('home_runs_scored', 'mean'),
    away_runs_mean=('away_runs_scored', 'mean'),
    hits_mean=('hits', 'mean'),
    home_runs_hit_mean=('home_runs_hit', 'mean'),
    walks_mean=('walks', 'mean'),
    strikeouts_mean=('strikeouts', 'mean'),
    temp_mean=('temp_f', 'mean'),
    rhum_mean=('rhum', 'mean'),
    pres_mean=('pres', 'mean'),
    wspd_mean=('wspd_mph', 'mean')
).round(3)

day_night_summary.index.name = 'Game Type'
day_night_summary

In [ ]:
# ============================================================
# Main Day/Night Diagnostic Figure
# ============================================================
order = ['Day', 'Night']
x = np.arange(len(order))
summary = day_night_summary.loc[order].copy()

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

axes[0, 0].bar(x - 0.25, summary['total_runs_mean'], width=0.25, color='#d62728', label='Total runs')
axes[0, 0].bar(x, summary['home_runs_mean'], width=0.25, color='#1f77b4', label='Home runs')
axes[0, 0].bar(x + 0.25, summary['away_runs_mean'], width=0.25, color='#ff7f0e', label='Away runs')
axes[0, 0].set_title('Run Production by Time of Day')
axes[0, 0].set_ylabel('Runs')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(order)
axes[0, 0].legend()

axes[0, 1].boxplot([team_df.loc[team_df['time_of_day'] == label, 'total_runs'].dropna().values for label in order], labels=order, patch_artist=True,
                   boxprops=dict(facecolor='#f2a5a0', alpha=0.7), medianprops=dict(color='black', linewidth=2))
axes[0, 1].set_title('Total Runs Distribution: Day vs Night')
axes[0, 1].set_ylabel('Total runs')

axes[1, 0].bar(x - 0.15, summary['temp_mean'], width=0.15, color='#f28e2b', label='Temperature')
axes[1, 0].bar(x, summary['rhum_mean'], width=0.15, color='#76b7b2', label='Humidity')
axes[1, 0].bar(x + 0.15, summary['wspd_mean'], width=0.15, color='#59a14f', label='Wind speed')
axes[1, 0].set_title('Weather Means by Time of Day')
axes[1, 0].set_ylabel('Weather value')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(order)
axes[1, 0].legend()

axes[1, 1].boxplot([team_df.loc[team_df['time_of_day'] == label, 'pres'].dropna().values for label in order], labels=order, patch_artist=True,
                   boxprops=dict(facecolor='#bab0ab', alpha=0.8), medianprops=dict(color='black', linewidth=2))
axes[1, 1].set_title('Pressure Distribution: Day vs Night')
axes[1, 1].set_ylabel('Pressure (hPa)')

for ax in axes.flat:
    ax.grid(alpha=0.25)

plt.suptitle('Minute Maid Park: Day/Night Scoring and Weather Context', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Weather Quintile Summaries
# ============================================================
def summarize_weather_bins(frame, bin_col, weather_col):
    summary = frame.groupby(bin_col, observed=True).agg(
        games=('game_pk', 'size'),
        weather_mean=(weather_col, 'mean'),
        total_runs_mean=('total_runs', 'mean'),
        home_runs_mean=('home_runs_scored', 'mean'),
        away_runs_mean=('away_runs_scored', 'mean'),
        hits_mean=('hits', 'mean'),
        strikeouts_mean=('strikeouts', 'mean')
    ).round(3)
    summary.index = summary.index.astype(str)
    return summary

temp_summary = summarize_weather_bins(team_df, 'temp_bin', 'temp_f')
rhum_summary = summarize_weather_bins(team_df, 'rhum_bin', 'rhum')
pres_summary = summarize_weather_bins(team_df, 'pres_bin', 'pres')
wspd_summary = summarize_weather_bins(team_df, 'wspd_bin', 'wspd_mph')

display(temp_summary)
display(rhum_summary)
display(pres_summary)
display(wspd_summary)

In [ ]:
# ============================================================
# Monthly Pattern: Are Day and Night Games Happening in Different Weather Windows?
# ============================================================
monthly_summary = team_df.groupby(['month_name', 'time_of_day'], observed=True).agg(
    games=('game_pk', 'size'),
    total_runs_mean=('total_runs', 'mean'),
    temp_mean=('temp_f', 'mean'),
    rhum_mean=('rhum', 'mean'),
    pres_mean=('pres', 'mean'),
    wspd_mean=('wspd_mph', 'mean')
).reset_index()

month_order = [m for m in ['Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct'] if m in monthly_summary['month_name'].unique()]
runs_pivot = monthly_summary.pivot(index='month_name', columns='time_of_day', values='total_runs_mean').reindex(month_order)
temp_pivot = monthly_summary.pivot(index='month_name', columns='time_of_day', values='temp_mean').reindex(month_order)
rhum_pivot = monthly_summary.pivot(index='month_name', columns='time_of_day', values='rhum_mean').reindex(month_order)
pres_pivot = monthly_summary.pivot(index='month_name', columns='time_of_day', values='pres_mean').reindex(month_order)
wspd_pivot = monthly_summary.pivot(index='month_name', columns='time_of_day', values='wspd_mean').reindex(month_order)

fig, axes = plt.subplots(3, 2, figsize=(16, 14))
runs_pivot.plot(kind='bar', ax=axes[0, 0], color=['#4c78a8', '#e45756'])
axes[0, 0].set_title('Monthly Total Runs: Day vs Night')
axes[0, 0].legend(title='Game Type')

temp_pivot.plot(kind='bar', ax=axes[0, 1], color=['#f28e2b', '#edc948'])
axes[0, 1].set_title('Monthly Temperature: Day vs Night')
axes[0, 1].legend(title='Game Type')

rhum_pivot.plot(kind='bar', ax=axes[1, 0], color=['#76b7b2', '#59a14f'])
axes[1, 0].set_title('Monthly Humidity: Day vs Night')
axes[1, 0].legend(title='Game Type')

pres_pivot.plot(kind='bar', ax=axes[1, 1], color=['#bab0ab', '#9c755f'])
axes[1, 1].set_title('Monthly Pressure: Day vs Night')
axes[1, 1].legend(title='Game Type')

wspd_pivot.plot(kind='bar', ax=axes[2, 0], color=['#86bc86', '#2f6b2f'])
axes[2, 0].set_title('Monthly Wind Speed: Day vs Night')
axes[2, 0].legend(title='Game Type')

monthly_games = monthly_summary.pivot(index='month_name', columns='time_of_day', values='games').reindex(month_order)
monthly_games.plot(kind='bar', ax=axes[2, 1], color=['#4c78a8', '#e45756'])
axes[2, 1].set_title('Monthly Game Counts: Day vs Night')
axes[2, 1].legend(title='Game Type')

for ax in axes.flat:
    ax.grid(axis='y', alpha=0.25)

plt.suptitle('Minute Maid Park: Seasonal Composition of Day and Night Games', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Conditional Day/Night Comparisons Within Weather Quintiles
# ============================================================
def conditional_day_night(frame, bin_col, label_name):
    grouped = frame.groupby([bin_col, 'time_of_day'], observed=True).agg(
        games=('game_pk', 'size'),
        total_runs_mean=('total_runs', 'mean'),
        home_runs_mean=('home_runs_scored', 'mean'),
        away_runs_mean=('away_runs_scored', 'mean')
    ).reset_index()
    grouped[label_name] = grouped[bin_col].astype(str)
    pivot = grouped.pivot(index=label_name, columns='time_of_day', values='total_runs_mean')
    pivot['Night_minus_Day'] = pivot.get('Night', np.nan) - pivot.get('Day', np.nan)
    return grouped, pivot

temp_grouped, temp_pivot = conditional_day_night(team_df, 'temp_bin', 'temp_label')
rhum_grouped, rhum_pivot = conditional_day_night(team_df, 'rhum_bin', 'rhum_label')
pres_grouped, pres_pivot = conditional_day_night(team_df, 'pres_bin', 'pres_label')
wspd_grouped, wspd_pivot = conditional_day_night(team_df, 'wspd_bin', 'wspd_label')

fig, axes = plt.subplots(4, 2, figsize=(18, 18))

temp_pivot[['Day', 'Night']].plot(kind='bar', ax=axes[0, 0], color=['#4c78a8', '#e45756'])
axes[0, 0].set_title('Total Runs by Temperature Quintile')
temp_pivot['Night_minus_Day'].plot(kind='bar', ax=axes[0, 1], color='#9467bd')
axes[0, 1].axhline(0, color='gray', linestyle='--', linewidth=1)
axes[0, 1].set_title('Night Minus Day: Temperature Quintiles')

rhum_pivot[['Day', 'Night']].plot(kind='bar', ax=axes[1, 0], color=['#4c78a8', '#e45756'])
axes[1, 0].set_title('Total Runs by Humidity Quintile')
rhum_pivot['Night_minus_Day'].plot(kind='bar', ax=axes[1, 1], color='#2ca02c')
axes[1, 1].axhline(0, color='gray', linestyle='--', linewidth=1)
axes[1, 1].set_title('Night Minus Day: Humidity Quintiles')

pres_pivot[['Day', 'Night']].plot(kind='bar', ax=axes[2, 0], color=['#4c78a8', '#e45756'])
axes[2, 0].set_title('Total Runs by Pressure Quintile')
pres_pivot['Night_minus_Day'].plot(kind='bar', ax=axes[2, 1], color='#8c564b')
axes[2, 1].axhline(0, color='gray', linestyle='--', linewidth=1)
axes[2, 1].set_title('Night Minus Day: Pressure Quintiles')

wspd_pivot[['Day', 'Night']].plot(kind='bar', ax=axes[3, 0], color=['#4c78a8', '#e45756'])
axes[3, 0].set_title('Total Runs by Wind-Speed Quintile')
wspd_pivot['Night_minus_Day'].plot(kind='bar', ax=axes[3, 1], color='#1f9a8a')
axes[3, 1].axhline(0, color='gray', linestyle='--', linewidth=1)
axes[3, 1].set_title('Night Minus Day: Wind-Speed Quintiles')

for ax in axes.flat:
    ax.grid(axis='y', alpha=0.25)

plt.suptitle('Minute Maid Park: Does the Day/Night Gap Survive Within Similar Weather Conditions?', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

display(temp_pivot)
display(rhum_pivot)
display(pres_pivot)
display(wspd_pivot)